## Creating research and writing agents using crewai 

In [1]:
from dotenv import dotenv_values
import os 
from openai import OpenAI

In [2]:
import sys
# sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/17.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/24.0.0/lib/python3.12/site-packages')

In [3]:
config = dotenv_values(".env")

# os.environ["OPENAI_API_KEY"] = config['OPENAI_API_KEY']
os.environ["NVIDIA_API_KEY"] = config['NVIDIA_API_KEY']
# os.environ["GEMINI_API_KEY"] = config['GEMINI_API_KEY']
# os.environ["DEEPSEEK_API_KEY"] = config['DEEPSEEK_API_KEY']

In [4]:
# config['OPENAI_API_KEY']

In [5]:
# client = OpenAI(
#     api_key=os.environ["GEMINI_API_KEY"], 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
)


print("client initialized")

client initialized


In [6]:
# os.environ["OPENAI_MODEL_NAME"] = 'gemini-3.1-flash-lite'
model_name = 'nvidia/nemotron-3-nano-30b-a3b:free'
os.environ["OPENAI_MODEL_NAME"] = model_name

In [7]:
# warning control
import warnings
warnings.filterwarnings('ignore')

In [8]:
from crewai import Agent, Task, Crew, LLM
# , LLM
# from langchain_litellm import ChatLiteLLM

In [10]:
# llm = LLM(
#     model="gemini/gemini-3.1-flash-lite",
#     api_key=os.environ["GEMINI_API_KEY"]
# )

llm = LLM(
    model=model_name,
    api_key=os.environ["NVIDIA_API_KEY"],
    custom_llm_provider="openrouter",
)

# llm = ChatLiteLLM(
#     model="gemini/gemini-3.1-flash-lite",
#     api_key=os.environ["GEMINI_API_KEY"]
# )

In [11]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
    llm=llm,
	verbose=False
)

In [12]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
    llm=llm, 
	verbose=False
)

In [13]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [14]:
import crewai_tools
print(crewai_tools.__version__)
print(dir(crewai_tools))

1.14.4
['AIMindTool', 'ApifyActorsTool', 'ArxivPaperTool', 'BedrockInvokeAgentTool', 'BedrockKBRetrieverTool', 'BraveImageSearchTool', 'BraveLLMContextTool', 'BraveLocalPOIsDescriptionTool', 'BraveLocalPOIsTool', 'BraveNewsSearchTool', 'BraveSearchTool', 'BraveVideoSearchTool', 'BraveWebSearchTool', 'BrightDataDatasetTool', 'BrightDataSearchTool', 'BrightDataWebUnlockerTool', 'BrowserbaseLoadTool', 'CSVSearchTool', 'CodeDocsSearchTool', 'ComposioTool', 'ContextualAICreateAgentTool', 'ContextualAIParseTool', 'ContextualAIQueryTool', 'ContextualAIRerankTool', 'CouchbaseFTSVectorSearchTool', 'CrewaiPlatformTools', 'DOCXSearchTool', 'DallETool', 'DatabricksQueryTool', 'DaytonaExecTool', 'DaytonaFileTool', 'DaytonaPythonTool', 'DirectoryReadTool', 'DirectorySearchTool', 'E2BExecTool', 'E2BFileTool', 'E2BPythonTool', 'EXASearchTool', 'EnterpriseActionTool', 'FileCompressorTool', 'FileReadTool', 'FileWriterTool', 'FirecrawlCrawlWebsiteTool', 'FirecrawlScrapeWebsiteTool', 'FirecrawlSearchTool'

In [15]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)

In [16]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

In [17]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)

In [18]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=False
)

In [19]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 341578f3-b7d7-41a1-b965-7dfda35d1514                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide     │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│  ID: 268a1354-99e6-4685-9ffa-46ae7c720804                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide     │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi there! 👋 Thanks so much for reaching out - it's an absolute pleasure to help DeepLearningAI with such an   │
│  important question! I completely understand why memory setup is crucial for your CrewAI workflows, especially  │
│  with Andrew Ng's team relying on this functionality.                                                           │
│                                                                                                                 │
│  Based on the **official CrewAI documentation** (which I accessed using our internal knowledge base), here's    │
│  exactly how to add memory to your Crew:                                                                        │
│                                                                                                                 │
│  ### 🧠 Memory Setup Guide for Your Crew                                                                        │
│  1. **Basic Memory Enablement**                                                                                 │
│     Simply add `memory=True` to your Crew configuration:                                                        │
│     ```python                                                                                                   │
│     crew = Crew(                                                                                                │
│         agents=[your_agent1, your_agent2],                                                                      │
│         tasks=[your_task1, your_task2],                                                                         │
│         memory=True,  # ✨ This enables memory by default!                                                      │
│         process='sequential'                                                                                    │
│     )                                                                                                           │
│     ```                                                                                                         │
│     *This automatically retains context within a single task execution chain.*                                  │
│                                                                                                                 │
│  2. **Long-Term Memory Configuration**                                                                          │
│     For **persistent memory across multiple sessions**, you'll need to:                                         │
│     - Choose a vector database backend (like Pinecone, Chroma, or Weaviate)                                     │
│     - Configure it using parameters like:                                                                       │
│       ```python                                                                                                 │
│       memory_backend='pinecone',                                                                                │
│       pinecone_api_key='your_api_key',                                                                          │
│       pinecone_index='memory_index'                                                                             │
│       ```                                                                                                       │
│     - Adjust retention settings with `memory_threshold` (d

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide     │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Task: Review the response drafted by the Senior Support Representative for DeepLearningAI's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hey there!👋 Thanks for reaching out – we love helping DeepLearningAI get the most out of CrewAI. 🎉           │
│                                                                                                                 │
│  Below is a step‑by‑step rundown (straight from our official **CrewAI Memory Management Guide**) that covers    │
│  everything you need to know to add memory to your Crew, whether you just need short‑term context within a      │
│  single run or long‑term persistence across multiple sessions.                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ Enable Memory for a Single Execution Chain                                                               │
│                                                                                                                 │
│  The easiest way to start is to set `memory=True` on your Crew definition:                                      │
│                                                                                                                 │
│  ```python                                                                                                      │
│  crew = Crew(                                                                                                   │
│      agents=[agent_one, agent_two],                                                                             │
│      tasks=[task_one, task_two],                                                                                │
│      memory=True,          # <-- turns on memory automatically                                                  │
│      process='sequential' # optional, but often useful                                                          │
│  )                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│  - **What it does:** The memory module keeps the conversation (or task) context alive **only for the duration   │
│  of that run**.  - **Why it’s handy:** You don’t have to manage any extra state – CrewAI handles it for you.    │
│                                                                                                                 │
│  > *Source: CrewAI Memory Management Guide – “Memory functionality is enabled by default when `memory=True` is  │
│  set.”*                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2️⃣ Persist Memory Across Multiple Sessions  If you 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for DeepLearningAI's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 341578f3-b7d7-41a1-b965-7dfda35d1514                                                                       │
│  Final Output: Hey there!👋 Thanks for reaching out – we love helping DeepLearningAI get the most out of        │
│  CrewAI. 🎉                                                                                                     │
│                                                                                                                 │
│  Below is a step‑by‑step rundown (straight from our official **CrewAI Memory Management Guide**) that covers    │
│  everything you need to know to add memory to your Crew, whether you just need short‑term context within a      │
│  single run or long‑term persistence across multiple sessions.                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ Enable Memory for a Single Execution Chain                                                               │
│                                                                                                                 │
│  The easiest way to start is to set `memory=True` on your Crew definition:                                      │
│                                                                                                                 │
│  ```python                                                                                                      │
│  crew = Crew(                                                                                                   │
│      agents=[agent_one, agent_two],                                                                             │
│      tasks=[task_one, task_two],                                                                                │
│      memory=True,          # <-- turns on memory automatically                                                  │
│      process='sequential' # optional, but often useful                                                          │
│  )                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│  - **What it does:** The memory module keeps the conversation (or task) context alive **only for the duration   │
│  of that run**.  - **Why it’s handy:** You don’t have to manage any extra state – CrewAI handles it for you.    │
│                                                                                                                 │
│  > *Source: CrewAI Memory Management Guide – “Memory functionality is enabled by default when `memory=True` is  │
│  set.”*                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                        

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
from IPython.display import Markdown
Markdown(result.raw)

Hey there!👋 Thanks for reaching out – we love helping DeepLearningAI get the most out of CrewAI. 🎉  

Below is a step‑by‑step rundown (straight from our official **CrewAI Memory Management Guide**) that covers everything you need to know to add memory to your Crew, whether you just need short‑term context within a single run or long‑term persistence across multiple sessions.

---

## 1️⃣ Enable Memory for a Single Execution Chain  

The easiest way to start is to set `memory=True` on your Crew definition:

```python
crew = Crew(
    agents=[agent_one, agent_two],
    tasks=[task_one, task_two],
    memory=True,          # <-- turns on memory automatically
    process='sequential' # optional, but often useful
)
```

- **What it does:** The memory module keeps the conversation (or task) context alive **only for the duration of that run**.  - **Why it’s handy:** You don’t have to manage any extra state – CrewAI handles it for you.

> *Source: CrewAI Memory Management Guide – “Memory functionality is enabled by default when `memory=True` is set.”*  

---

## 2️⃣ Persist Memory Across Multiple Sessions  If you need the Crew to remember things **between different runs** (e.g., across days or runs triggered by external events), you’ll have to point it at a vector database. Here’s a minimal, copy‑and‑paste‑ready configuration:

```python
crew = Crew(
    agents=[agent_one, agent_two],
    tasks=[task_one, task_two],
    memory=True,
    memory_backend='pinecone',               # choose Pinecone, Chroma, or Weaviate
    pinecone_api_key='YOUR_PINECONE_API_KEY',
    pinecone_index='memory_index',
    memory_threshold=0.9                    # optional: how “similar” a piece of context must be to be kept
)
```

### Quick checklist for the vector‑DB route
| Step | What you need |
|------|---------------|
| **Pick a DB** | Pinecone, Chroma, or Weaviate are officially supported. |
| **Create an API key** | – Pinecone: generate in the Pinecone console.<br>– Chroma/Weaviate: generate per their docs. |
| **Create an index** | Typically a single index named `memory_index` works, but you can name it whatever you like. |
| **Set `memory_threshold`** | Default is `0.8`. Raising it (e.g., to `0.9`) keeps only the most relevant snippets. |
| **Store your API keys securely** | Use environment variables (`os.getenv('PINECONE_API_KEY')`) rather than hard‑coding. |

> *Source: CrewAI Memory Management Guide – “For production‑grade persistent memory, integrate with vector databases following our dedicated integration guide.”*  

---

## 3️⃣ Handy Pro Tips from Our Crew  

| Tip | Why it helps |
|-----|--------------|
| **Start simple** – enable `memory=True` first, test with a tiny context window (e.g., 2‑3 messages). | Makes debugging painless. |
| **Tweak `memory_threshold`** – 0.8–0.9 is a good range; higher values = stricter relevance. | Prevents “noise” from irrelevant past turns. |
| **Chunk your context** – if you’re feeding long documents, split them into logical chunks before adding to memory. | Keeps each chunk semantically meaningful. |
| **Check the log output** – CrewAI prints a short “Memory snapshot” at the end of each run. | Lets you see exactly what was retained. |
| **Clear memory when needed** – you can call `crew.memory.clear()` manually if you want a fresh slate. | Useful for testing new configurations. |

---

## 4️⃣ Need a Full Working Example?  

We can spin up a **complete, runnable script** that:

1. Sets up two agents (e.g., a Planner & an Executor).  
2. Connects to Pinecone with your API key.  
3. Executes a sample task while persisting context across multiple calls.  

Just let us know if you’d like that code snippet – we’ll tailor it to the exact agents and tasks you’re using.

---

## 5️⃣ Want a Deep‑Dive Session?  

If you’d prefer a live walkthrough with one of our **Solutions Architects**, we can schedule a 30‑minute call. They’ll:

- Walk through your specific workflow.  
- Help you pick the best vector DB and configuration tweaks.  
- Answer any performance or scaling questions you have.  Just reply with a few times that work for you, and we’ll lock it in.

---

### TL;DR  
- **Short‑term memory:** `memory=True` (no extra config).  
- **Long‑term memory:** Choose a vector DB, supply API key & index, optionally set `memory_threshold`.  
- All details are documented in the **[CrewAI Memory Management Guide](https://docs.crewai.com/en/enterprise/guides/memory)** and the linked integration guide.

---

Feel free to ping us with any follow‑up questions, share your current Crew config, or just say “let’s see that code example.” We’re super excited to help DeepLearningAI make the most of CrewAI’s memory powers! 🚀  Warm regards,  
Your CrewAI Support Team 🌟  

*P.S. Andrew Ng’s team has been an absolute delight to work with – we’re thrilled to support such an impactful AI education initiative!*

In [ ]:
# topic = "What power is based on Game of Thrones"
# result = crew.kickoff(inputs={"topic": topic})

In [ ]:
# Markdown(result.raw)